###  Démarrage de la session Spark

In [ ]:
# L'exécution de cette cellule démarre l'application Spark

Starting Spark application


<u>Affichage des informations sur la session en cours et liens vers Spark UI</u> :

In [ ]:
%%info

In [ ]:
"""
Ce script permet l'extraction des features à partir des images.
Réduction PCA à 50 dimensions.

Sauvegarde des résultats :
En Parquet partitionné (PATH_Result)
En Parquet consolidé dans un sous-dossier /parquet (PATH_Result_consolidated/parquet)
En CSV consolidé dans un sous-dossier /csv (PATH_Result_consolidated/csv)
"""

###  Installation des packages

Les packages nécessaires ont été installé via l'étape de **bootstrap** à l'instanciation du serveur.

### Import des librairies

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, element_at, split, pandas_udf, PandasUDFType
from pyspark.sql.types import ArrayType, FloatType
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
import numpy as np
import io

### Définition des PATH pour charger les images et enregistrer les résultats

Nous accédons directement à nos **données sur S3** comme si elles étaient **stockées localement**.

In [ ]:
PATH = "s3a://fruits-p8-oc/"
PATH_Data = PATH + "data/Test_cloud"
PATH_Result = PATH + "data/Output_cloud"
PATH_Result_consolidated = PATH + "data/Output_cloud_consolidated"


In [ ]:
# Affiche les chemins configurés
print(f"PATH:        {PATH}\nPATH_Data:   {PATH_Data}\nPATH_Result: {PATH_Result}")

### Traitement des données

#### Chargement des données

In [ ]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

In [ ]:
images.show(5)

<u>Je ne conserve que le **path** de l'image et j'ajoute <br />
    une colonne contenant les **labels** de chaque image</u> :

In [ ]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

# Préparation du modèle

# Importation des librairies

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, element_at, split, pandas_udf, PandasUDFType, udf
from pyspark.sql.types import ArrayType, FloatType, StringType
from pyspark.ml.feature import PCA
from pyspark.ml.linalg import Vectors, VectorUDT
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from PIL import Image
import io
import numpy as np
import pandas as pd

## Configuration de la session Spark

In [ ]:
# ---------------------------------------- #
# 1. Configuration de la session Spark     #
# ---------------------------------------- #
spark = SparkSession.builder \
    .appName("Image_Features_PCA") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

## Initialisation du modèle MobileNetV2 

In [ ]:
# ---------------------------------------- #
# 2. Initialisation du modèle MobileNetV2  #
# ---------------------------------------- #
model = MobileNetV2(weights="imagenet", include_top=True, input_shape=(224, 224, 3))
new_model = Model(inputs=model.input, outputs=model.layers[-2].output)
broadcast_weights = spark.sparkContext.broadcast(new_model.get_weights())

def model_fn():
    model = MobileNetV2(weights=None, include_top=True, input_shape=(224, 224, 3))
    new_model = Model(inputs=model.input, outputs=model.layers[-2].output)
    new_model.set_weights(broadcast_weights.value)
    return new_model

## Prétraitement des images 

In [ ]:
# ---------------------------------------- #
# 3. Prétraitement des images              #
# ---------------------------------------- #
def preprocess(content):
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

## Définition du processus de chargement des images et application de leur featurisation à travers l'utilisation de pandas UDF¶

In [ ]:
# ---------------------------------------- #
# 4. Extraction des features               #
# ---------------------------------------- #
def featurize_series(model, content_series):
    inputs = np.stack(content_series.map(preprocess))
    preds = model.predict(inputs)
    return pd.Series([p.flatten() for p in preds])

@pandas_udf(ArrayType(FloatType()), PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)


#### Exécutions des actions d'extractions de features

In [ ]:
# ---------------------------------------- #
# 5. Extraction des features avec UDF      #
# ---------------------------------------- #
features_df = images.repartition(24).select(
    col("path"),
    col("label"),
    featurize_udf("content").alias("features")
)

# ---------------------------------------- #
# 6. Conversion des features en VectorUDT  #
# ---------------------------------------- #
def to_vector(features):
    return Vectors.dense(features)

to_vector_udf = udf(to_vector, VectorUDT())
features_df = features_df.withColumn("features_vector", to_vector_udf(col("features")))


## Réduction de dimension PCA

In [ ]:

# ---------------------------------------- #
# 7. Réduction PCA à 50 dimensions         #
# ---------------------------------------- #
pca = PCA(k=50, inputCol="features_vector", outputCol="pcaFeatures")
pca_model = pca.fit(features_df)
features_with_pca = pca_model.transform(features_df)

# ---------------------------------------- #
# 8. Conversion des pcaFeatures            #
# ---------------------------------------- #
@udf(ArrayType(FloatType()))
def vector_to_array(vector):
    return vector.toArray().tolist() if vector is not None else None

@udf(StringType())
def array_to_string(array):
    return ",".join([str(x) for x in array]) if array else None

features_with_pca = features_with_pca.withColumn("pcaFeatures_array", vector_to_array(col("pcaFeatures")))
features_with_pca_csv = features_with_pca.withColumn("pcaFeatures_string", array_to_string(col("pcaFeatures_array")))


In [ ]:

# ---------------------------------------- #
# 9. Sauvegarde des résultats             #
# ---------------------------------------- #
# Sauvegarder en fichiers Parquet partitionnés
features_with_pca.write.mode("overwrite").parquet(PATH_Result)

# 1. Sauvegarder le Parquet consolidé avec un nom explicite
features_with_pca.coalesce(1).write.mode("overwrite").parquet(PATH_Result_consolidated + "/parquet")

# 2. Sauvegarder le CSV consolidé avec un nom explicite
features_with_pca_csv.select("path", "label", "pcaFeatures_string") \
    .coalesce(1).write.mode("overwrite").csv(PATH_Result_consolidated + "/csv", header=True)



### Chargement des données enregistrées et validation du résultat

# Lecture du Parquet consolidé :

In [ ]:
import boto3
import pandas as pd
import io
import s3fs
import logging
import numpy as np

In [ ]:
def find_valid_file(s3fs_obj, path, suffix):
    """
    Trouve le premier fichier valide (qui correspond au suffixe) dans un dossier S3.
    """
    files = s3fs_obj.ls(path)
    for file in files:
        if file.endswith(suffix):
            return f"s3://{file}"
    raise FileNotFoundError(f"Aucun fichier valide trouvé dans {path} avec le suffixe {suffix}")

# Trouver les fichiers CSV et Parquet
csv_file = find_valid_file(fs, f"{PATH_Result_consolidated}/csv", ".csv")
parquet_file = find_valid_file(fs, f"{PATH_Result_consolidated}/parquet", ".parquet")

print(f"Fichier CSV trouvé : {csv_file}")
print(f"Fichier Parquet trouvé : {parquet_file}")


In [ ]:
logging.disable(logging.CRITICAL)

# Initialise S3FileSystem avec le bon client_kwargs
fs = s3fs.S3FileSystem(client_kwargs={'region_name': 'eu-west-1'})

# Fonction pour trouver automatiquement un fichier valide
def find_valid_file(s3fs_obj, path, suffix):
    """
    Trouve le premier fichier valide (qui correspond au suffixe) dans un dossier S3.
    """
    files = s3fs_obj.ls(path)
    for file in files:
        if file.endswith(suffix) and "_SUCCESS" not in file:  # Ignore _SUCCESS
            return f"s3://{file}"
    raise FileNotFoundError(f"Aucun fichier valide trouvé dans {path} avec le suffixe {suffix}")

# Chemin du dossier CSV consolidé dans S3
PATH_Result_consolidated = "s3://fruits-p8-oc/data/Output_cloud_consolidated"

try:
    # Trouver le chemin du fichier CSV valide
    csv_file_path = find_valid_file(fs, f"{PATH_Result_consolidated}/csv", ".csv")
    print(f"Fichier CSV sélectionné : {csv_file_path}")
    
    # Lire le fichier CSV depuis S3
    with fs.open(csv_file_path, mode='rb') as file:
        # Charge le DataFrame
        df = pd.read_csv(file)
        print("Dimensions du DataFrame complet :", df.shape)
        
        # Convertir la colonne 'pcaFeatures_string' en listes de floats
        df['pcaFeatures_list'] = df['pcaFeatures_string'].apply(lambda x: list(map(float, x.split(','))))
        
        # Imprime la dimension des PCA features pour la première ligne
        print("Dimension des PCA features (exemple ligne 0) :", len(df.loc[0, 'pcaFeatures_list']))
        
        # Vérifie si toutes les lignes ont la même dimension
        pca_dimensions = df['pcaFeatures_list'].apply(len).unique()
        print("Dimensions uniques des PCA features dans le DataFrame :", pca_dimensions)

    # Afficher un aperçu des données
    print("Aperçu du DataFrame :")
    print(df.head())

except Exception as e:
    print(f"Erreur lors de la lecture du fichier : {e}")


In [ ]:
# Lis le fichier Parquet consolidé
df_parquet = pd.read_parquet(parquet_file, engine="pyarrow")

print("Aperçu du fichier Parquet :")
print(df_parquet.head())

# Vérifie la shape après PCA (nombre de dimensions)
print(f"Shape des PCA features : {len(df_parquet.loc[0, 'pcaFeatures_array'])}")


In [ ]:
len(df.loc[0, 'pcaFeatures_list'])

In [ ]:
len(df_parquet.loc[0, "pcaFeatures_array"])

In [ ]:
#La PCA a bien fonctionnée, le Parquet renvoie 50 pour len(df_parquet.loc[0, "pcaFeatures_array"]).
#Le CSV renvoie également 50 pour len(df_csv.loc[0, "pcaFeatures_list"]).